# Manual Completo PySpark

Guia pratico de referencia para PySpark 3.5. Todos os exemplos usam dados criados em memoria, sem depender de storage externo.

**Indice:**
1. SparkSession
2. Criar DataFrames
3. Inspecionar Dados
4. Select e Alias
5. Filter / Where
6. withColumn (adicionar/modificar)
7. Renomear e Remover Colunas
8. Ordenar, Limitar e Amostrar
9. Distinct e Duplicatas
10. Funcoes de String
11. Funcoes Numericas
12. Funcoes de Data
13. Tratamento de Nulls
14. Cast (conversao de tipos)
15. Agregacoes e GroupBy
16. Pivot
17. Joins
18. Window Functions
19. SQL Puro
20. UDFs
21. Schemas e Tipos de Dados
22. Cache e Performance
23. Referencia Rapida

---

## 1. SparkSession

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("manual-pyspark") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} | Master: {spark.sparkContext.master}")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/26 23:03:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 3.5.5 | Master: spark://spark-master:7077


---
## 2. Criar DataFrames

In [2]:
# A partir de lista de tuplas
dados = [
    (1, "Alice", 30, "Engenharia", 8500.0),
    (2, "Bruno", 25, "Marketing", 5200.0),
    (3, "Carla", 35, "Engenharia", 9800.0),
    (4, "Diego", 28, "Vendas", 6100.0),
    (5, "Elena", 32, "Marketing", 7300.0),
    (6, "Felipe", 27, "Engenharia", 7000.0),
    (7, "Gabi", 40, "Vendas", 8900.0),
    (8, "Hugo", 22, "Marketing", 4500.0),
    (9, "Ivan", 29, "Vendas", 5800.0),
    (10, "Julia", 33, "Engenharia", 9200.0),
]

df = spark.createDataFrame(dados, ["id", "nome", "idade", "departamento", "salario"])
df.show()

+---+------+-----+------------+-------+
| id|  nome|idade|departamento|salario|
+---+------+-----+------------+-------+
|  1| Alice|   30|  Engenharia| 8500.0|
|  2| Bruno|   25|   Marketing| 5200.0|
|  3| Carla|   35|  Engenharia| 9800.0|
|  4| Diego|   28|      Vendas| 6100.0|
|  5| Elena|   32|   Marketing| 7300.0|
|  6|Felipe|   27|  Engenharia| 7000.0|
|  7|  Gabi|   40|      Vendas| 8900.0|
|  8|  Hugo|   22|   Marketing| 4500.0|
|  9|  Ivan|   29|      Vendas| 5800.0|
| 10| Julia|   33|  Engenharia| 9200.0|
+---+------+-----+------------+-------+



In [3]:
# A partir de Pandas
import pandas as pd

pdf = pd.DataFrame({
    "produto": ["Notebook", "Mouse", "Teclado", "Monitor", "Webcam"],
    "preco": [4500.0, 89.90, 250.0, 1200.0, 350.0],
    "estoque": [10, 150, 80, 25, 60]
})

df_produtos = spark.createDataFrame(pdf)
df_produtos.show()

+--------+------+-------+
| produto| preco|estoque|
+--------+------+-------+
|Notebook|4500.0|     10|
|   Mouse|  89.9|    150|
| Teclado| 250.0|     80|
| Monitor|1200.0|     25|
|  Webcam| 350.0|     60|
+--------+------+-------+



In [4]:
# DataFrame vazio com schema
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

schema = StructType([
    StructField("id", IntegerType()),
    StructField("nome", StringType())
])

df_vazio = spark.createDataFrame([], schema)
print(f"Linhas: {df_vazio.count()}")
df_vazio.printSchema()

Linhas: 0
root
 |-- id: integer (nullable = true)
 |-- nome: string (nullable = true)



---
## 3. Inspecionar Dados

In [ ]:
# Schema
df.printSchema()

# Tipos
print(df.dtypes)

# Colunas
print(df.columns)

# Dimensoes
print(f"Linhas: {df.count()} | Colunas: {len(df.columns)}")

In [ ]:
# Formas de exibir
df.show()                        # 20 primeiras (padrao)
df.show(5)                       # 5 primeiras
df.show(5, truncate=False)       # sem truncar texto
df.show(vertical=True)           # formato vertical

In [ ]:
# Estatisticas descritivas
df.describe().show()

# Estatisticas detalhadas
df.summary().show()

---
## 4. Select e Alias

In [ ]:
from pyspark.sql.functions import col, lit, upper

# Selecionar colunas
df.select("nome", "idade").show()

# Com alias
df.select(
    col("nome").alias("funcionario"),
    col("idade"),
    col("departamento").alias("depto")
).show()

# selectExpr — expressoes SQL inline
df.selectExpr(
    "nome",
    "idade",
    "salario * 12 as salario_anual",
    "UPPER(departamento) as depto"
).show()

---
## 5. Filter / Where

In [ ]:
# Comparacao simples
df.filter(col("idade") > 30).show()

# String SQL
df.where("idade > 30 AND salario > 8000").show()

# AND
df.filter((col("idade") > 25) & (col("departamento") == "Engenharia")).show()

# OR
df.filter((col("departamento") == "Vendas") | (col("departamento") == "Marketing")).show()

# NOT
df.filter(~(col("departamento") == "Engenharia")).show()

In [ ]:
# IN
df.filter(col("departamento").isin("Vendas", "Marketing")).show()

# LIKE / contains / startswith / endswith
df.filter(col("nome").startswith("A")).show()
df.filter(col("nome").contains("li")).show()
df.filter(col("nome").endswith("a")).show()

# BETWEEN
df.filter(col("salario").between(6000, 8000)).show()

# IS NULL / IS NOT NULL
df.filter(col("nome").isNotNull()).show()

---
## 6. withColumn (adicionar/modificar)

In [ ]:
from pyspark.sql.functions import when, lit, upper, concat

# Valor fixo
df.withColumn("empresa", lit("MinhaEmpresa")).show()

# Modificar existente
df.withColumn("nome", upper(col("nome"))).show()

# Coluna calculada
df.withColumn("salario_anual", col("salario") * 12).show()

# CASE WHEN
df.withColumn("nivel",
    when(col("salario") < 6000, "Junior")
    .when(col("salario") < 8000, "Pleno")
    .otherwise("Senior")
).show()

In [ ]:
# Multiplas colunas de uma vez
df.withColumn("ano_nascimento", lit(2026) - col("idade")) \
  .withColumn("bonus", col("salario") * 0.1) \
  .withColumn("salario_com_bonus", col("salario") + col("salario") * 0.1) \
  .show()

---
## 7. Renomear e Remover Colunas

In [ ]:
# Renomear uma coluna
df.withColumnRenamed("nome", "funcionario").show(5)

# Renomear varias
mapa = {"nome": "funcionario", "departamento": "depto", "salario": "remuneracao"}
df_r = df
for antigo, novo in mapa.items():
    df_r = df_r.withColumnRenamed(antigo, novo)
df_r.show(5)

# Remover coluna
df.drop("departamento").show(5)

# Remover varias
df.drop("departamento", "id").show(5)

---
## 8. Ordenar, Limitar e Amostrar

In [ ]:
from pyspark.sql.functions import asc, desc

# Ascendente
df.orderBy("idade").show()

# Descendente
df.orderBy(col("salario").desc()).show()

# Multiplas colunas
df.orderBy(col("departamento").asc(), col("salario").desc()).show()

# Limitar
df.limit(3).show()

# Amostra aleatoria
df.sample(fraction=0.5, seed=42).show()

---
## 9. Distinct e Duplicatas

In [ ]:
# Valores distintos
df.select("departamento").distinct().show()
print(f"Departamentos unicos: {df.select('departamento').distinct().count()}")

# Remover duplicatas completas
df.dropDuplicates().show()

# Remover duplicatas por colunas especificas
df.dropDuplicates(["departamento"]).show()

---
## 10. Funcoes de String

In [5]:
from pyspark.sql.functions import (
    upper, lower, trim, initcap,
    length, substring, concat, concat_ws,
    regexp_replace, regexp_extract, split,
    lpad, rpad, reverse
)

df.select(
    col("nome"),
    upper(col("nome")).alias("maiusculo"),
    lower(col("nome")).alias("minusculo"),
    initcap(col("nome")).alias("capitalizado"),
    length(col("nome")).alias("tamanho"),
    substring(col("nome"), 1, 3).alias("3_chars"),
    reverse(col("nome")).alias("reverso"),
    lpad(col("nome"), 10, "*").alias("pad_esq")
).show(truncate=False)

NameError: name 'col' is not defined

26/02/27 01:31:02 WARN StandaloneAppClient$ClientEndpoint: Connection to spark-master:7077 failed; waiting for master to reconnect...
26/02/27 01:31:02 WARN StandaloneSchedulerBackend: Disconnected from Spark cluster! Waiting for reconnection...
26/02/27 01:31:02 WARN StandaloneAppClient$ClientEndpoint: Connection to spark-master:7077 failed; waiting for master to reconnect...


In [ ]:
# Concatenar
df.select(
    concat(col("nome"), lit(" - "), col("departamento")).alias("completo"),
    concat_ws(" | ", col("nome"), col("departamento")).alias("com_pipe")
).show(truncate=False)

# Substituir com regex
df.select(
    col("departamento"),
    regexp_replace(col("departamento"), "Engenharia", "TI").alias("substituido")
).distinct().show()

# Split em array
df.select(
    col("nome"),
    split(col("nome"), "a").alias("partes")
).show(truncate=False)

---
## 11. Funcoes Numericas

In [ ]:
from pyspark.sql.functions import (
    abs as spark_abs, ceil, floor, round as spark_round,
    sqrt, pow as spark_pow, greatest, least
)

df_num = spark.createDataFrame([
    (1, 15.7, -3.2),
    (2, 8.3, 4.5),
    (3, 22.1, -1.8),
], ["id", "valor_a", "valor_b"])

df_num.select(
    col("id"),
    spark_round(col("valor_a"), 0).alias("arredondado"),
    ceil(col("valor_a")).alias("teto"),
    floor(col("valor_a")).alias("piso"),
    spark_abs(col("valor_b")).alias("absoluto"),
    sqrt(col("valor_a")).alias("raiz"),
    spark_pow(col("valor_a"), 2).alias("quadrado"),
    greatest(col("valor_a"), col("valor_b")).alias("maior"),
    least(col("valor_a"), col("valor_b")).alias("menor")
).show()

---
## 12. Funcoes de Data

In [ ]:
from pyspark.sql.functions import (
    current_date, current_timestamp, date_format,
    year, month, dayofmonth, dayofweek, hour,
    datediff, months_between, add_months, date_add, date_sub,
    to_date, to_timestamp, from_unixtime
)

df_datas = spark.createDataFrame([
    (1, "2024-03-15", "2024-06-20 14:30:00", 1710500000),
    (2, "2023-11-01", "2025-01-10 09:15:00", 1698800000),
], ["id", "data_str", "timestamp_str", "epoch"])

# Converter strings para datas
df_datas.select(
    col("id"),
    to_date(col("data_str")).alias("data"),
    to_timestamp(col("timestamp_str")).alias("timestamp"),
    from_unixtime(col("epoch")).alias("epoch_convertido"),
    current_date().alias("hoje"),
    current_timestamp().alias("agora")
).show(truncate=False)

In [ ]:
# Extrair partes
df_datas.select(
    to_date(col("data_str")).alias("data"),
    year(to_date(col("data_str"))).alias("ano"),
    month(to_date(col("data_str"))).alias("mes"),
    dayofmonth(to_date(col("data_str"))).alias("dia"),
    dayofweek(to_date(col("data_str"))).alias("dia_semana"),
    date_format(to_date(col("data_str")), "dd/MM/yyyy").alias("formato_br"),
    date_format(to_date(col("data_str")), "EEEE").alias("nome_dia")
).show(truncate=False)

# Aritmetica
df_datas.select(
    to_date(col("data_str")).alias("data"),
    date_add(to_date(col("data_str")), 30).alias("+30_dias"),
    date_sub(to_date(col("data_str")), 7).alias("-7_dias"),
    add_months(to_date(col("data_str")), 3).alias("+3_meses"),
    datediff(current_date(), to_date(col("data_str"))).alias("dias_atras"),
    months_between(current_date(), to_date(col("data_str"))).alias("meses_atras")
).show(truncate=False)

---
## 13. Tratamento de Nulls

In [ ]:
from pyspark.sql.functions import coalesce

df_nulos = spark.createDataFrame([
    (1, "Alice", 8500.0),
    (2, None, 5200.0),
    (3, "Carla", None),
    (4, None, None),
], ["id", "nome", "salario"])

print("Dados originais:")
df_nulos.show()

# Filtrar nulos
print("Somente com nome nulo:")
df_nulos.filter(col("nome").isNull()).show()

# Substituir nulos
print("fillna:")
df_nulos.fillna({"nome": "Desconhecido", "salario": 0.0}).show()

# coalesce — primeiro valor nao-nulo
print("coalesce:")
df_nulos.select(
    col("id"),
    coalesce(col("nome"), lit("SEM NOME")).alias("nome")
).show()

# Remover linhas com null
print("dropna (qualquer null):")
df_nulos.dropna().show()

print("dropna (null em 'nome'):")
df_nulos.dropna(subset=["nome"]).show()

---
## 14. Cast (conversao de tipos)

In [ ]:
df.select(
    col("idade").cast("string").alias("str"),
    col("idade").cast("double").alias("double"),
    col("salario").cast("int").alias("int"),
    col("salario").cast("decimal(10,2)").alias("decimal")
).printSchema()

# Tipos: string, int, long, float, double, boolean,
#        date, timestamp, decimal(p,s), array, map

---
## 15. Agregacoes e GroupBy

In [ ]:
from pyspark.sql.functions import (
    count, countDistinct, sum as spark_sum, avg,
    min as spark_min, max as spark_max,
    first, last, collect_list, collect_set,
    stddev, variance
)

# Agregacao global
df.select(
    count("*").alias("total"),
    countDistinct("departamento").alias("deptos"),
    avg("salario").alias("salario_medio"),
    spark_min("salario").alias("menor_salario"),
    spark_max("salario").alias("maior_salario"),
    spark_sum("salario").alias("folha_total"),
    stddev("salario").alias("desvio_padrao")
).show()

In [ ]:
# GroupBy
df.groupBy("departamento").agg(
    count("*").alias("total"),
    avg("salario").alias("media_salario"),
    spark_min("salario").alias("menor"),
    spark_max("salario").alias("maior"),
    collect_list("nome").alias("nomes")
).orderBy(desc("total")).show(truncate=False)

In [ ]:
# GroupBy com multiplas colunas
df_vendas = spark.createDataFrame([
    ("SP", "Eletronicos", 1500), ("SP", "Eletronicos", 2300),
    ("SP", "Moveis", 800),       ("RJ", "Eletronicos", 1200),
    ("RJ", "Moveis", 950),       ("RJ", "Moveis", 1100),
    ("MG", "Eletronicos", 900),  ("MG", "Moveis", 750),
], ["estado", "categoria", "valor"])

df_vendas.groupBy("estado", "categoria").agg(
    count("*").alias("qtd"),
    spark_sum("valor").alias("total"),
    avg("valor").alias("media")
).orderBy("estado", "categoria").show()

---
## 16. Pivot

In [ ]:
# Pivot — transforma valores de linha em colunas
df_vendas.groupBy("estado") \
    .pivot("categoria") \
    .agg(spark_sum("valor")) \
    .show()

# Pivot com lista fixa (mais performatico)
df_vendas.groupBy("estado") \
    .pivot("categoria", ["Eletronicos", "Moveis"]) \
    .agg(spark_sum("valor")) \
    .show()

---
## 17. Joins

In [ ]:
df_func = spark.createDataFrame([
    (1, "Alice", 101),
    (2, "Bruno", 102),
    (3, "Carla", 101),
    (4, "Diego", 103),
    (5, "Elena", None),
], ["id", "nome", "depto_id"])

df_deptos = spark.createDataFrame([
    (101, "Engenharia", "SP"),
    (102, "Marketing", "RJ"),
    (103, "Vendas", "MG"),
    (104, "RH", "SP"),
], ["depto_id", "depto_nome", "estado"])

print("=== Funcionarios ===")
df_func.show()
print("=== Departamentos ===")
df_deptos.show()

In [ ]:
# INNER — so com match nos dois lados
print("=== INNER ===")
df_func.join(df_deptos, "depto_id", "inner").show()

# LEFT — todos da esquerda + match da direita
print("=== LEFT ===")
df_func.join(df_deptos, "depto_id", "left").show()

# RIGHT — todos da direita + match da esquerda
print("=== RIGHT ===")
df_func.join(df_deptos, "depto_id", "right").show()

# FULL — todos de ambos
print("=== FULL ===")
df_func.join(df_deptos, "depto_id", "full").show()

In [ ]:
# LEFT ANTI — esquerda SEM match na direita
print("=== LEFT ANTI (sem departamento) ===")
df_func.join(df_deptos, "depto_id", "left_anti").show()

# LEFT SEMI — esquerda COM match (sem colunas da direita)
print("=== LEFT SEMI (com departamento, sem colunas extra) ===")
df_func.join(df_deptos, "depto_id", "left_semi").show()

In [ ]:
# Join com nomes de coluna diferentes
df_a = spark.createDataFrame([(1, "Alice"), (2, "Bruno")], ["user_id", "nome"])
df_b = spark.createDataFrame([(1, 100), (2, 200)], ["id_usuario", "score"])

df_a.join(df_b, df_a.user_id == df_b.id_usuario, "inner") \
    .drop("id_usuario") \
    .show()

# Join com multiplas condicoes
# df_a.join(df_b, (df_a.col1 == df_b.col1) & (df_a.col2 == df_b.col2), "inner")

---
## 18. Window Functions

Calcular valores com base em um grupo de linhas **sem colapsar** (diferente do groupBy).

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, rank, dense_rank, lag, lead

# Ranking por departamento (maior salario primeiro)
w = Window.partitionBy("departamento").orderBy(col("salario").desc())

df.select(
    col("*"),
    row_number().over(w).alias("posicao"),     # 1,2,3 (sem repetir)
    rank().over(w).alias("rank"),               # 1,2,2,4 (empate pula)
    dense_rank().over(w).alias("dense_rank"),   # 1,2,2,3 (empate nao pula)
).show()

In [ ]:
# Lag / Lead — valor anterior / proximo
w_ord = Window.partitionBy("departamento").orderBy("salario")

df.select(
    col("nome"),
    col("departamento"),
    col("salario"),
    lag("salario", 1).over(w_ord).alias("salario_anterior"),
    lead("salario", 1).over(w_ord).alias("salario_proximo"),
    (col("salario") - lag("salario", 1).over(w_ord)).alias("diferenca")
).show()

In [ ]:
# Agregacoes cumulativas
w_depto = Window.partitionBy("departamento")
w_acum = Window.partitionBy("departamento").orderBy("salario").rowsBetween(Window.unboundedPreceding, 0)

df.select(
    col("nome"),
    col("departamento"),
    col("salario"),
    spark_sum("salario").over(w_acum).alias("acumulado"),
    avg("salario").over(w_acum).alias("media_acum"),
    spark_sum("salario").over(w_depto).alias("total_depto"),
    (col("salario") / spark_sum("salario").over(w_depto) * 100).alias("pct_depto")
).show()

---
## 19. SQL Puro

In [ ]:
# Registrar como view
df.createOrReplaceTempView("funcionarios")

# SELECT
spark.sql("SELECT * FROM funcionarios WHERE idade > 30").show()

# Agregacao
spark.sql("""
    SELECT departamento,
           COUNT(*) as total,
           ROUND(AVG(salario), 2) as media_salario,
           MIN(salario) as menor,
           MAX(salario) as maior
    FROM funcionarios
    GROUP BY departamento
    ORDER BY total DESC
""").show()

# Subquery
spark.sql("""
    SELECT nome, salario
    FROM funcionarios
    WHERE salario > (SELECT AVG(salario) FROM funcionarios)
    ORDER BY salario DESC
""").show()

# Window function
spark.sql("""
    SELECT nome, departamento, salario,
           ROW_NUMBER() OVER (PARTITION BY departamento ORDER BY salario DESC) as ranking
    FROM funcionarios
""").show()

---
## 20. UDFs (User Defined Functions)

In [ ]:
from pyspark.sql.functions import udf, pandas_udf
from pyspark.sql.types import StringType, FloatType

# UDF com decorator
@udf(returnType=StringType())
def classificar_salario(salario):
    if salario is None:
        return "N/A"
    elif salario < 6000:
        return "Junior"
    elif salario < 8000:
        return "Pleno"
    else:
        return "Senior"

df.select("nome", "salario", classificar_salario("salario").alias("nivel")).show()

In [ ]:
# Pandas UDF (mais performatico — usa Apache Arrow)
@pandas_udf(FloatType())
def normalizar(serie: pd.Series) -> pd.Series:
    return (serie - serie.min()) / (serie.max() - serie.min())

df.select("nome", "salario", normalizar("salario").alias("salario_norm")).show()

---
## 21. Schemas e Tipos de Dados

In [ ]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, LongType,
    FloatType, DoubleType, DecimalType,
    BooleanType, DateType, TimestampType,
    ArrayType, MapType
)

# Schema explicito (mais seguro e rapido que inferSchema)
schema = StructType([
    StructField("id", IntegerType(), nullable=False),
    StructField("nome", StringType(), nullable=False),
    StructField("salario", DecimalType(10, 2), nullable=True),
    StructField("ativo", BooleanType(), nullable=True),
    StructField("tags", ArrayType(StringType()), nullable=True),
])

# DDL string (alternativa curta)
# schema = "id INT, nome STRING, salario DECIMAL(10,2), ativo BOOLEAN"

# Tabela de tipos
tipos = spark.createDataFrame([
    ("StringType", "Texto", "'hello'"),
    ("IntegerType", "Inteiro 32-bit", "42"),
    ("LongType", "Inteiro 64-bit", "9999999999"),
    ("FloatType", "Decimal 32-bit", "3.14"),
    ("DoubleType", "Decimal 64-bit", "3.14159265"),
    ("DecimalType(10,2)", "Decimal preciso", "12345.67"),
    ("BooleanType", "Booleano", "true/false"),
    ("DateType", "Data", "2024-03-15"),
    ("TimestampType", "Data+hora", "2024-03-15 14:30:00"),
    ("ArrayType", "Lista", "[1, 2, 3]"),
    ("MapType", "Dicionario", "{chave: valor}"),
], ["tipo", "descricao", "exemplo"])
tipos.show(truncate=False)

---
## 22. Cache e Performance

In [ ]:
# Cache — manter na memoria para reutilizacao
df_cached = df.cache()
df_cached.count()  # dispara o cache

# Agora e mais rapido
df_cached.filter(col("idade") > 30).show()
df_cached.groupBy("departamento").count().show()

# Liberar
df_cached.unpersist()
print("Cache liberado")

In [ ]:
# Particoes
print(f"Particoes atuais: {df.rdd.getNumPartitions()}")

# repartition(n) — redistribui com shuffle (para AUMENTAR)
print(f"repartition(4): {df.repartition(4).rdd.getNumPartitions()}")

# coalesce(n) — reduz SEM shuffle (para DIMINUIR)
print(f"coalesce(1): {df.coalesce(1).rdd.getNumPartitions()}")

In [ ]:
# Explain — ver plano de execucao
df.filter(col("idade") > 30) \
  .groupBy("departamento") \
  .agg(avg("salario").alias("media")) \
  .explain(True)

In [ ]:
# Broadcast join — para tabelas pequenas (evita shuffle)
from pyspark.sql.functions import broadcast

# Spark envia a tabela pequena para todos os executors
df_func.join(broadcast(df_deptos), "depto_id", "inner").show()

---
## 23. Referencia Rapida

| Operacao | Codigo |
|----------|--------|
| Criar sessao | `SparkSession.builder.appName("x").getOrCreate()` |
| Criar DF | `spark.createDataFrame(dados, ["col1", "col2"])` |
| Selecionar | `df.select("col1", "col2")` |
| Filtrar | `df.filter(col("x") > 10)` |
| Adicionar coluna | `df.withColumn("nova", expr)` |
| Renomear | `df.withColumnRenamed("old", "new")` |
| Remover coluna | `df.drop("col")` |
| Ordenar | `df.orderBy(col("x").desc())` |
| Agrupar | `df.groupBy("col").agg(count("*"))` |
| Pivot | `df.groupBy("a").pivot("b").agg(sum("c"))` |
| Join | `df1.join(df2, "key", "inner")` |
| Distinct | `df.dropDuplicates(["col"])` |
| Nulos | `df.fillna({"col": valor})` |
| SQL | `spark.sql("SELECT * FROM view")` |
| CASE WHEN | `when(cond, val).otherwise(val)` |
| Window | `func().over(Window.partitionBy("x"))` |
| Cache | `df.cache()` / `df.unpersist()` |
| Schema | `df.printSchema()` |
| Contar | `df.count()` |
| Particoes | `df.rdd.getNumPartitions()` |
| Explain | `df.explain(True)` |

In [ ]:
spark.stop()
print("SparkSession encerrada.")